# AI-3 Session 1.2 — Proving Improvement: Evaluation Framework via Phoenix

Last session you built two retrieval techniques — HyDE and enrichment — and you could *see* they worked. Today we build the machinery that tells you whether an improvement is **real**.

**What we're building today:**
- A curated **golden set** pushed to Phoenix as a versioned dataset
- Two evaluators — `retrieval_hit` (Pattern A, deterministic) and `answer_addresses_question` (Pattern B, LLM-as-judge)
- Two Phoenix **experiments** — naive vs HyDE, side-by-side in the UI

**By the end of class you'll be able to say:**
*"HyDE wins on N-of-10 queries, naive wins on M-of-10 — here are the exact queries where each one fails."*

Vibes out. Evidence in.

In [ ]:
# Path setup — notebook is in notebooks/1_2/, pipeline is at repo root
import sys, os
from pathlib import Path

repo_root = str(Path().resolve().parent.parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)  # So .env and chroma_db paths resolve

from dotenv import load_dotenv
load_dotenv()
print(f"Working from: {os.getcwd()}")

### Phoenix Stack Reset — Run ONLY if imports fail below

If the next cell (imports) errors with something like:

```
ModuleNotFoundError: No module named 'phoenix.evals.models'
```

— your `.venv` has incompatible Phoenix package versions from incremental installs. `arize-phoenix`, `arize-phoenix-client`, and `arize-phoenix-evals` overlap the `phoenix.*` namespace, and a mismatched set leaves the experiments module pointing at a path that doesn't exist in the installed `phoenix.evals`. A clean rebuild of `.venv` from `pyproject.toml` resolves it.

**If the import cell works — skip the reset cell. Move on.**

**If it fails — do this:**

1. Run the reset cell below (~60 seconds)
2. **Restart the kernel**: Kernel → Restart
3. Re-run Cell 1 (path setup), then the imports cell

This is a one-time fix per machine — once your `.venv` is clean, you won't need it again.

In [ ]:
# Phoenix stack reset — ONLY run if the imports cell below fails.
# Rebuilds .venv from pyproject.toml. Takes ~60 seconds.
# After this completes: RESTART THE KERNEL (Kernel → Restart), then re-run Cell 1.

!rm -rf .venv uv.lock && uv sync

In [ ]:
# Imports — Phoenix client, our eval infrastructure, Anthropic
from phoenix.client import Client
from pipeline.eval.golden_set import GOLDEN_SET
from pipeline.eval.tasks import naive_task
import anthropic

print("All imports OK!")

In [ ]:
# Verify everything is wired
print(f"Golden set: {len(GOLDEN_SET)} queries")

client = Client()
print("Phoenix: connected")

# Quick Claude sanity check
anthropic_client = anthropic.Anthropic()
resp = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=30,
    messages=[{"role": "user", "content": "Say 'ready' in one word."}],
)
print(f"Claude: {resp.content[0].text.strip()}")

### All systems green — let's go

If any check above failed, fix it before continuing:

1. **Phoenix connection failed** — check `PHOENIX_API_KEY`, `PHOENIX_COLLECTOR_ENDPOINT`, `PHOENIX_PROJECT_NAME` in `.env`
2. **Golden set length wrong** — pull `main`, you have an older version of `golden_set.py`
3. **Claude call failed** — check `ANTHROPIC_API_KEY`

## The 1.1 Correction — DR-012 Revisit

Before we build today's evaluation machinery, we need to revisit something from Session 1.1.

The original `enrich_and_store()` we shipped had a bug. It stored **one row per chunk**, with an embedding of the *combined* (chunk text + questions) string. That sounds reasonable — both signals in one vector — but it smeared answer-space and question-space together and diluted the question-space matching that was the whole point of enrichment.

**DR-012 fix** (already in your repo if you pulled `main`):

- Embed **pure question text** — no chunk text concatenated
- Store **one row per question**, not one per chunk
- `enriched_retrieve` **over-fetches** by `N_QUESTIONS_PER_CHUNK` and **dedupes** back to unique chunks

The `documents` field (what we show on retrieval) still points to the chunk text. The `embeddings` field (what we search on) is the pure question vector. They are **decoupled by design** — that is the core of the fix.

This is pedagogy, not damage control. The instructor ran the scorecard, the scorecard caught the bug, the fix was a storage-contract change. That cycle — run the numbers, read the numbers, fix the code — is what we are about to build the infrastructure for.

In [ ]:
# See the evidence: run all three strategies on one query and observe the differences.
# This is a preview — by end of class, you'll have a full scorecard across all 10 golden queries.
from pipeline.retrieval.naive import naive_retrieve
from pipeline.retrieval.hyde import hyde_retrieve
from pipeline.retrieval.enriched import enriched_retrieve

query = "What happened at the Q4 board meeting?"
print(f"Query: {query}\n")

print("=== NAIVE ===")
for r in naive_retrieve(query, top_k=3):
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")

print("\n=== HyDE ===")
for r in hyde_retrieve(query, n_results=3):
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")

print("\n=== ENRICHED (DR-012 fix) ===")
for r in enriched_retrieve(query, n_results=3):
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")

## PCA Callback — Seeing the Gap Close

Open `notebooks/embedding_space_viz.ipynb` (the instructor will drive on the projector).

**What you'll see in that notebook:**

1. **Blue dots — answer chunks.** Your raw corpus, clustered in answer space.
2. **Orange dots — raw user questions.** Clustered in question space. The gap between blue and orange is the embedding gap from AI-2 3.2.
3. **Purple dots — HyDE vectors.** Hypothetical answers for the 10 golden questions. They land in **answer cluster** — HyDE moves the query into the neighborhood where real answers live.
4. **Green dots — enrichment vectors (post DR-012).** One per generated question. They land in **question cluster**, right next to user queries — questions match questions.
5. **Yellow dots (optional) — pre-DR-012 enrichment.** Combined-text embeddings floating between clusters, belonging to neither cleanly. The bug, visible in geometry.

**Two takeaways:**

- HyDE and enrichment attack the gap from **opposite sides** — HyDE moves the query, enrichment moves the index.
- An embedding bug is **visible in PCA space** if you look. The scorecard catches it first; the geometry explains why the fix works.

That is the last time today we talk about geometry. The rest of the session is about the evaluation machinery that catches bugs like this **automatically** — so you don't have to squint at 3D scatter plots to know your pipeline is wrong.

## What's Old, What's New

You already know most of evaluation. Let me name it.

| Concept | Where you met it | What's new in 1.2 |
|---|---|---|
| Test case (question + expected + pass/fail) | AI-1 2.2 | Lives in a **persistent** Phoenix dataset, not a notebook variable |
| Two-pipeline comparison | AI-2 3.2 | Becomes a **side-by-side experiment** in the Phoenix UI |
| Phoenix observability | AI-2 4.1 | Datasets + experiments + traces, **all in one place** |
| **Evaluator function** | NEW | A function bound to `output` + `expected` that returns a score |
| **LLM-as-judge** | NEW | An evaluator that uses Claude to grade answer quality |

**Three things that are actually new today:**

1. **Persistence** — the golden set is a versioned dataset (`northbrook_golden_v1`), not a notebook cell
2. **Regression** — any two experiments can be compared, any time
3. **Phoenix as a platform** — datasets + evaluators + experiments, not just traces

The first primitive you build today: an **evaluator function**. Before that — the dataset it binds to.

## Golden Set Walkthrough

We provide a 10-query set in `pipeline/eval/golden_set.py`. Each entry has the same shape:

```python
{
    "id": "vacation_policy",              # stable identifier
    "question": "What is the vacation policy?",
    "expected_answer": "Eligible full-time employees receive 20 vacation days...",
    "expected_source": ["vacation_policy_2025.md"],  # list — multi-doc queries may have several
    "category": "policy_lookup",          # for slicing in the UI
    "difficulty": "easy",
}
```

- `question` → goes to the pipeline
- `expected_answer` → feeds the LLM judge (Pattern B)
- `expected_source` → feeds `retrieval_hit` (Pattern A)
- `category`, `difficulty` → metadata for slicing results in the UI

**Why we provide it and don't have you write it today:** authoring a good golden set is 90% of Lab 2's rubric. Today's skill is *using* one via Phoenix. Clean separation.

In [ ]:
# Inspect the first 3 entries
for q in GOLDEN_SET[:3]:
    print(f"[{q['id']}] ({q['category']}, {q['difficulty']})")
    print(f"  Q: {q['question']}")
    print(f"  expected_source: {q['expected_source']}")
    print()

In [ ]:
# Category breakdown — what coverage does the set provide?
from collections import Counter

categories = Counter(q["category"] for q in GOLDEN_SET)
difficulties = Counter(q["difficulty"] for q in GOLDEN_SET)

print("Categories:")
for cat, n in categories.most_common():
    print(f"  {cat}: {n}")

print("\nDifficulties:")
for diff, n in difficulties.most_common():
    print(f"  {diff}: {n}")

## Push the Golden Set to Phoenix

Phoenix wants three **parallel lists**, keyed by position:

- `inputs[i]` → what goes to your pipeline: `{"question": "..."}`
- `outputs[i]` → what evaluators see as `expected`: `{"expected_answer": "...", "expected_source": [...]}`
- `metadata[i]` → data *about* the case (not handed to the pipeline): `{"id": "...", "category": "...", "difficulty": "..."}`

`client.datasets.create_dataset()` uploads all three as one **named, versioned** dataset. Once it's up, every experiment today binds to the same name.

You can run the script directly (`uv run python scripts/push_golden_set.py`), but let's do it inline so you see each piece.

In [ ]:
# Build the three parallel lists and push the dataset
inputs = [{"question": q["question"]} for q in GOLDEN_SET]

outputs = [
    {
        "expected_answer": q["expected_answer"],
        "expected_source": q["expected_source"],
    }
    for q in GOLDEN_SET
]

metadata = [
    {
        "id": q["id"],
        "category": q["category"],
        "difficulty": q["difficulty"],
    }
    for q in GOLDEN_SET
]

# Upload — this will error if the name already exists. That's intentional:
# named datasets are supposed to be stable. If you already pushed, skip this cell.
dataset = client.datasets.create_dataset(
    name="northbrook_golden_v1",
    dataset_description="Seed golden set for AI-3 RAG evaluation. Grows in Lab 2.",
    inputs=inputs,
    outputs=outputs,
    metadata=metadata,
)

print(f"Uploaded {len(GOLDEN_SET)} queries as dataset 'northbrook_golden_v1'")
print("View at: https://app.phoenix.arize.com (Datasets tab)")

### Phoenix UI Tour — Verify the Dataset

In your browser, go to **https://app.phoenix.arize.com** and:

1. Click **Datasets** in the left sidebar
2. Find `northbrook_golden_v1` — 10 rows
3. Click into it — you should see three columns: `input` (question), `output` (expected_answer + expected_source), `metadata` (id, category, difficulty)
4. Any experiment you run later today will show up under this dataset's **Experiments** tab

**If you see `DatasetAlreadyExists`:** good — you already ran this. No need to push twice.

**If you see `Unauthorized`:** your `PHOENIX_API_KEY` is not loading. Check `.env` is in the repo root and `load_dotenv()` is called.

### Annotation Configs — Instructor Demo (5 min)

Quick detour. Phoenix has a **third eval path** besides automated evaluators: **human annotations** (thumbs-up / thumbs-down).

The instructor will demo on the projector:

1. Navigate to **Settings → Annotation Configs**
2. Click **Create Config**, name it `thumbs_up_down`, type `categorical`, values `thumbs_up` and `thumbs_down`
3. Save

Once the config exists, any span, experiment result, or dataset row has an **Annotate** button. Those annotations live in the same observability system as your automated scores.

**Why this matters later:** in Session 3.2 (feedback loops), the thumbs-up button in your Streamlit UI calls the Phoenix API to write one of these. Today's job is awareness — the human path exists. Implementation comes later.

## Pattern A — `retrieval_hit` (Deterministic Evaluator)

The simplest possible evaluator: a boolean function that returns True if retrieval pulled at least one of the expected source files.

**Why this evaluator is worth having:**

- **Deterministic** — no LLM, no randomness. Same input, same score, every time.
- **Cheap** — zero API cost. Runs on every experiment for free.
- **Focused** — one question, one boolean. Did retrieval find the right file? Yes or no.
- **Independent of generation** — grades retrieval *without* caring about answer quality.

**Phoenix parameter binding** is the key concept here. Phoenix reads your function's parameter names and passes data by matching name:

- `input` → dataset's `inputs[i]` (the question)
- `output` → whatever the task returned (the `{chunks, answer}` dict)
- `expected` → dataset's `outputs[i]` (expected_answer + expected_source)
- `metadata` → dataset's `metadata[i]` (optional)

You only declare the parameters you actually use. We need `output` (for retrieved chunks) and `expected` (for the source list).

In [ ]:
# Pattern A: retrieval_hit — fill in the function body together.
# This is the same signature you see in pipeline/eval/evaluators.py.
# Once it works here, copy your implementation into that file.

def retrieval_hit(output: dict, expected: dict) -> bool:
    """Did retrieval pull at least one of the expected source documents?

    A deterministic evaluator — no LLM involved, just a set-membership check.
    This grades the RETRIEVAL half of the pipeline independently of the
    generation half.

    Args:
        output: What the task returned. Has a `chunks` key — a list of dicts
            with `metadata.source` for each retrieved chunk.
        expected: The ground-truth entry from the golden set. Has an
            `expected_source` key — a list of filenames that SHOULD be
            retrieved for this query.

    Returns:
        True if any expected source appears in the retrieved set.
    """
    # Step 1: Pull the list of retrieved sources from output["chunks"].
    #         Each chunk has source in chunk["metadata"]["source"].
    # Step 2: Pull the list of expected sources from expected["expected_source"].
    # Step 3: Return True if any expected source is in the retrieved list.
    pass

### Test `retrieval_hit` on one example

We're not running the full experiment yet — just sanity-checking the evaluator against one row. The shape of `input_row`, `expected`, and `output` below matches exactly what Phoenix will pass at experiment time.

In [ ]:
# Simulate one dataset row
input_row = {"question": "What is the vacation policy?"}
expected = {
    "expected_answer": (
        "Eligible full-time employees receive 20 vacation days per calendar year..."
    ),
    "expected_source": ["vacation_policy_2025.md"],
}

# Run the task — returns {question, chunks, answer}
output = naive_task(input_row)
retrieved_sources = [c["metadata"].get("source") for c in output["chunks"]]
print(f"Retrieved sources: {retrieved_sources}")

# Score it
hit = retrieval_hit(output, expected)
print(f"\nretrieval_hit: {hit}")
assert hit is True, "Expected True — vacation_policy_2025.md should be in retrieved"
print("Pattern A works!")

**What this evaluator does NOT do:**

- Does not check if the *right chunk* of the file was retrieved
- Does not check answer quality at all
- Does not care about score values — just membership

That is deliberate. **One evaluator, one concern.** When you see `retrieval_hit = 0.3` in the UI, you know exactly what that means: 3 of 10 queries pulled at least one right file. If the evaluator did five things, the score would be ambiguous.

## Pattern B — `answer_addresses_question` (LLM-as-Judge)

Retrieval hit tells you if the pipeline found the right file. But **finding the right file and generating a good answer from it are two different things**. You can retrieve perfectly and still hallucinate in the generation step. So you need an evaluator that reads the generated answer and grades it.

**Three options for grading a generated answer:**

| Option | Verdict |
|---|---|
| Exact string match | Brittle. Useless. |
| Substring check | Slightly less useless. Still brittle. |
| **LLM-as-judge** | Give question + expected + generated to another LLM, ask it to grade. |

We use **Claude Haiku 4.5** as the judge. It's cheap (~$0.001/call), plenty accurate for PASS/FAIL classification, and orders of magnitude cheaper than running Sonnet as a judge.

**Cost math:** 10 queries × 2 pipelines × 1 judge call per case = 20 judge calls ≈ $0.02 per student per experiment. You could run this 100 times a day for less than a coffee.

**Important:** the judge sees the **ground-truth expected answer**. It's not grading from scratch; it's comparing generated vs expected and deciding if they align.

### The Judge Prompt

The prompt matters. Three things to notice about the template below:

1. **Full context** — question, expected, generated, all three in the prompt. The judge has the whole picture.
2. **Constrained verdict format** — `PASS:` or `FAIL:` as the first line. Easy to parse, no ambiguity.
3. **Brief reason required** — forces the judge to justify its call, gives debuggable output when we scan results in the UI.

The question the judge is asked to answer has **two conjoined conditions**:

> *Does the generated answer substantively address the question AND align with the expected answer?*

Both must be true for PASS. A generated answer that addresses the question but is factually misaligned → FAIL. A generated answer that aligns with expected but rambles and never answers the question directly → FAIL. Both, or FAIL.

In [ ]:
# The judge prompt template — placeholders get filled in at call time.
JUDGE_TEMPLATE = """You are grading a RAG system's answer.

Question: {prompt}
Expected answer: {expected}
Generated answer: {response}

Does the generated answer substantively address the question
AND align with the expected answer?

Respond with exactly one line:
PASS: <brief reason>
or
FAIL: <brief reason>"""

In [ ]:
# Module-level client — the underscore signals "private to this evaluator,
# not part of the API Phoenix binds to." Only the function below is Phoenix-facing.
_client = anthropic.Anthropic()


def answer_addresses_question(input: dict, output: dict, expected: dict) -> bool:
    """Does the generated answer align with the expected answer?

    Uses Claude Haiku 4.5 as the judge. The judge sees the question, the
    ground-truth expected answer, AND the generated answer — then returns
    PASS or FAIL based on alignment.

    This is the "LLM-as-judge" pattern: cheaper and more scalable than human
    review, more semantic than string matching.

    Args:
        input: The dataset input dict. Has a `question` key.
        output: The task output dict. Has an `answer` key.
        expected: The ground-truth dict. Has an `expected_answer` key.

    Returns:
        True if the judge returns PASS, False otherwise.
    """
    # Step 1: Format JUDGE_TEMPLATE with the question, expected, and generated answer.
    #         Cap the generated answer at 2000 chars to control token cost.
    # Step 2: Call Claude Haiku 4.5 with max_tokens=100 (the judge only needs one short line).
    # Step 3: Parse the first line — if it starts with "PASS" (case-insensitive), return True.
    #         Otherwise return False.
    pass

### Test `answer_addresses_question` on one example

One judge call, ~1 second, returns True or False. If the verdict surprises you — read the reason. (Temporarily add `print(verdict)` inside the function to see what Haiku said.) **The judge is a tool; calibrate it, don't blindly trust it.**

In [ ]:
# Reuse input_row + expected from the Pattern A test
output = naive_task(input_row)
print(f"Generated answer:\n  {output['answer'][:300]}...\n")

verdict = answer_addresses_question(input_row, output, expected)
print(f"answer_addresses_question: {verdict}")

# Sanity test: a deliberately bad answer should FAIL
bad_output = {"chunks": [], "answer": "Flamingos are pink wading birds."}
bad_verdict = answer_addresses_question(input_row, bad_output, expected)
print(f"\nWith a nonsense answer: {bad_verdict}")
assert bad_verdict is False, "Judge should FAIL a clearly wrong answer"
print("Pattern B works!")

### Save your implementations back to `pipeline/eval/evaluators.py`

Both functions work. Now copy them into `pipeline/eval/evaluators.py` so the experiment runner can import them.

The file already has the signatures, docstrings, and `JUDGE_TEMPLATE` — you just need to replace the `pass` in each function body with your working implementation.

Once saved, the experiment runner script can import them:

```python
from pipeline.eval.evaluators import retrieval_hit, answer_addresses_question
```

In [ ]:
# Verify the module imports cleanly after you save
# (restart kernel if needed so the fresh module loads)
from pipeline.eval.evaluators import retrieval_hit as rh_module
from pipeline.eval.evaluators import answer_addresses_question as aaq_module

# Sanity: both should be callable and return the right types
sample_out = naive_task({"question": "What is the vacation policy?"})
sample_exp = {
    "expected_answer": "Eligible full-time employees receive 20 vacation days...",
    "expected_source": ["vacation_policy_2025.md"],
}

print(f"retrieval_hit: {rh_module(sample_out, sample_exp)}")
print(f"answer_addresses_question: {aaq_module({'question': 'What is the vacation policy?'}, sample_out, sample_exp)}")
print("\nModule evaluators ready!")

## Run the Experiment — Naive vs HyDE, Side-by-Side

Time to wire it all together. One `run_experiment()` call per pipeline — same dataset, same evaluators, different task. Phoenix logs both as **sibling experiments** under the dataset.

- **`dataset`** — the one we pushed earlier (`northbrook_golden_v1`)
- **`task`** — the function that takes a row's input and returns `{chunks, answer}`
- **`evaluators`** — a **list** of functions; Phoenix uses each function's `__name__` as the column label in the UI
- **`experiment_name`** — a label you pick so you can distinguish runs later
- **`experiment_metadata`** — free-form dict for filtering/grouping in the UI

You can also run the full script via `uv run python scripts/run_experiment.py` — same thing, unwrapped.

In [ ]:
# Wire everything up
from pipeline.eval.tasks import naive_task, hyde_task
from pipeline.eval.evaluators import retrieval_hit, answer_addresses_question

MODEL_NAME = "claude-sonnet-4-5"
EVALUATORS = [retrieval_hit, answer_addresses_question]

# Fetch the dataset we uploaded earlier
dataset = client.datasets.get_dataset(dataset="northbrook_golden_v1")
print(f"Dataset loaded: northbrook_golden_v1")

In [ ]:
# Run the naive experiment — ~45-60 seconds for 10 queries
print("Running: naive / claude-sonnet-4-5")
naive_experiment = client.experiments.run_experiment(
    dataset=dataset,
    task=naive_task,
    evaluators=EVALUATORS,
    experiment_name=f"naive / {MODEL_NAME}",
    experiment_metadata={"pipeline": "naive", "model": MODEL_NAME},
)
print("Done.")

In [ ]:
# Run the HyDE experiment — ~60-90 seconds (HyDE adds a Claude call per query)
print("Running: hyde / claude-sonnet-4-5")
hyde_experiment = client.experiments.run_experiment(
    dataset=dataset,
    task=hyde_task,
    evaluators=EVALUATORS,
    experiment_name=f"hyde / {MODEL_NAME}",
    experiment_metadata={"pipeline": "hyde", "model": MODEL_NAME},
)
print("Done.")

### Phoenix UI — Compare Results

Go to **https://app.phoenix.arize.com** → **Datasets** → **northbrook_golden_v1** → **Experiments** tab.

You should see two experiments: `naive / claude-sonnet-4-5` and `hyde / claude-sonnet-4-5`. Things to explore:

1. **Aggregate scores** — for each experiment, what's the pass rate on `retrieval_hit`? On `answer_addresses_question`?
2. **Per-row view** — click into one experiment. Find a row where `retrieval_hit` is False. Look at the retrieved sources vs the expected source — what was pulled instead?
3. **Judge reasons** — click a row where `answer_addresses_question` is False. The judge's reason is in the trace — use it to diagnose whether the generation hallucinated, rambled, or missed the question.
4. **Compare mode** — select both experiments and switch to compare. You get a side-by-side table: same query, two pipelines, two sets of scores.

**This is the payoff.** Two numbers that say "HyDE is better than naive — here, specifically, by this much, on these queries." Every pipeline change from now on gets graded by the same dataset and same evaluators. That is the contract.

## What You Built Today

- **`northbrook_golden_v1`** — a versioned Phoenix dataset with 10 curated queries
- **`retrieval_hit`** — Pattern A: a deterministic, zero-cost evaluator that grades retrieval independently of generation
- **`answer_addresses_question`** — Pattern B: an LLM-as-judge evaluator that grades generated answers against ground-truth expected answers
- **Two Phoenix experiments** — naive vs HyDE, the same machinery that will score every pipeline change for the rest of AI-3

## Before Next Session

Confirm all four:

1. Your `northbrook_golden_v1` dataset is visible in Phoenix
2. Your `pipeline/eval/evaluators.py` has both functions implemented
3. You ran the experiment at least once and saw both runs in the UI
4. You can click into a single row and understand what you are looking at

This code is the foundation of **Lab 2** and of every pipeline change you make for the rest of AI-3.

## Questions to Leave With

**On cost and benefit:** Every change you make from now on can be scored by this framework. What changes are worth running the experiment for? What changes are cheap enough to run on every pull request?

**On the judge:** Haiku is the judge today. What happens to your eval if the judge has its own biases or blindspots? How would you know?

**On the dataset's life:** Your golden set is 10 queries today. Lab 2 grows it. Where do new queries come from — your intuition, production logs, user feedback, all three? And how do you keep the set honest as it grows?

## Next Session — 2.1: Context Windows and Chat History

We start the Streamlit chat application. A new problem: the conversation **keeps growing**. After 20 turns, you're pushing the context window limit — and turn 21 might be "tell me more about that."

**Preview question to hold:** if turn 21 is "tell me more about that" — what does "that" refer to, and how do you figure it out *before* you retrieve?